In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, interact, fixed
import ipywidgets as widgets
import pickle

save_file_name = 'morpho2d1.pkl'
with open(save_file_name,'rb') as pk:
    x_list,y_list,u_list,M_list = pickle.load(pk)

In [2]:
def DeTri(nx,ny,nw):
    cellnum = len(nx)
    ut = np.triu(np.ones((cellnum, cellnum)), 1)
    id1 = np.ones((cellnum, 1)) * np.arange(1, cellnum + 1)
    id2 = id1.T
    id1 = id1[ut > 0].astype(int) - 1
    id2 = id2[ut > 0].astype(int) - 1
    nx1 = nx[id1]
    ny1 = ny[id1]
    nw1 = nw[id1]
    nx2 = nx[id2]
    ny2 = ny[id2]
    nw2 = nw[id2]
    dx = nx1 - nx2
    dy = ny1 - ny2
    dw = nw1**2 - nw2**2
    dl = np.sqrt(dx**2 + dy**2)
    mx = (nx1 + nx2) / 2
    my = (ny1 + ny2) / 2
    mx = mx - dw * dx / (2 * dl**2)
    my = my - dw * dy / (2 * dl**2)
    d0 = (mx - nx1)**2 + (my - ny1)**2 - nw1**2
    a = (dy*(nx.reshape(-1,1)-mx) - dx*(ny.reshape(-1,1)-my)) / dl
    b = (mx-nx.reshape(-1,1))**2 + (my-ny.reshape(-1,1))**2 - (nw.reshape(-1,1)**2)
    b = b - d0
    j1 = (id1 * len(id1)) + np.arange(len(id1))
    j2 = (id2 * len(id2)) + np.arange(len(id2))
    a.flat[j1] = 1
    a.flat[j2] = -1
    md = np.sqrt(np.maximum(-d0, 0))
    b.flat[j1] = md*2
    b.flat[j2] = md*2
    j0 = np.where(a==0)
    a[j0] = 1
    b[j0] = np.sign(b[j0])*np.max(md)*2
    c = b/a/2
    
    ap = (a > 0)
    am = (a < 0)
    cp = ap * c + (~ap) * md
    cm = am * c - (~am) * md
    cp0 = np.min(cp, axis=0)
    cm0 = np.max(cm, axis=0)
    jj = np.where(cp0 - cm0 > 0)[0]

    vx = dy[jj] / dl[jj]
    vy = -dx[jj] / dl[jj]
    x1 = mx[jj] + vx * cp0[jj]
    x2 = mx[jj] + vx * cm0[jj]
    y1 = my[jj] + vy * cp0[jj]
    y2 = my[jj] + vy * cm0[jj]
    xx = np.vstack((x1, x2))
    yy = np.vstack((y1, y2))
    ids = np.vstack((id1[jj], id2[jj]))

    return ids, xx, yy

In [3]:
def BoundaryCircle(nx,ny,nw):
    cellnum = len(nx)
    nx1 = np.tile(nx,(cellnum,1))
    ny1 = np.tile(ny,(cellnum,1))
    nw1 = np.tile(nw,(cellnum,1))
    nx2 = nx1.T
    ny2 = ny1.T
    nw2 = nw1.T
    dx = nx1-nx2
    dy = ny1-ny2
    dw = nw1**2-nw2**2
    dl = np.sqrt(dx**2+dy**2)+np.eye(cellnum);
    ds = (dl+dw/dl)/2
    dz = np.sqrt(np.maximum(nw1**2-ds**2,0))
    
    ag0 = np.arctan2(dy,dx)
    ag1 = np.arctan2(dz,ds)
    ag1 = ag1*(1-np.eye(cellnum))
    ags = np.tile(ag0,(6,1))+np.concatenate((-ag1-2*np.pi,-ag1,-ag1+2*np.pi,ag1-2*np.pi,ag1,ag1+2*np.pi))
    ags = np.minimum(np.maximum(ags,-np.pi),np.pi)
    bb = np.sort(ags,axis=0)
    cc = (np.argsort(ags,axis=0)>cellnum*3-1)
    e0 = np.arange(0, 6*cellnum-1, 2)
    
    arcend0 = np.array([])
    arcend1 = np.array([])
    arcid = np.array([])
    for ii in range(cellnum):
        c0 = np.where(cc[:,ii]==0)[0]
        c1 = np.where(cc[:,ii]==1)[0]
        c0i = np.where(c0-e0==0)[0]
        c1i = np.where(c1-e0==1)[0]
        c0 = c0[c0i]
        c1 = c1[c1i]
        b0 = bb[c0,ii]
        b1 = bb[c1,ii]
        jj = np.where(b1 - b0 > 0)[0]
        b1 = np.concatenate(([-np.pi], b1[jj]))
        b0 = np.concatenate((b0[jj], [np.pi]))
        jj = np.where(b1 - b0 < 0)[0]
        arcend0 = np.hstack((arcend0,b1[jj]))
        arcend1 = np.hstack((arcend1,b0[jj]))
        arcid = np.hstack((arcid,np.tile(ii,len(jj))))
    arcends = np.vstack((arcend0,arcend1))
    arcid = arcid.astype(int)
    
    return arcends, arcid

In [4]:
def PlotCells(nx,ny,nw,ids,xx,yy,arcends,arcid,xlim,ylim,dual):
    cellnum = len(nx)
    nxc = nx[arcid]
    nyc = ny[arcid]
    nwc = nw[arcid]
    darc = arcends[1,:]-arcends[0,:]
    ags = np.tile(darc,(100,1)) * np.tile(np.linspace(0,1,100),(len(darc),1)).T + arcends[0,:]
    xc = nxc - np.cos(ags)*nwc
    yc = nyc - np.sin(ags)*nwc
    plt.figure
    plt.plot(xx,yy,color='black')
    plt.plot(xc,yc,color='black')
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.gca().set_aspect('equal')
    if dual:
        plt.plot(nx[ids],ny[ids],color='blue')
    plt.show()

In [5]:
def MorphoShape(t,xlim,ylim,dual):
    nx = x_list[t]
    ny = y_list[t]
    nw = u_list[t]
    ids, xx, yy = DeTri(nx,ny,nw)
    arcends, arcid = BoundaryCircle(nx,ny,nw)
    PlotCells(nx,ny,nw,ids,xx,yy,arcends,arcid,xlim,ylim,dual)

interact(
    MorphoShape,
    t=widgets.IntSlider(min=0,max=len(x_list)-1,step=1,value=0),
    xlim=widgets.IntRangeSlider(min=-100,max=100,step=1,value=[-20,40]),
    ylim=widgets.IntRangeSlider(min=-100,max=100,step=1,value=[-20,40]),
    dual = widgets.Checkbox(value=False)
);

interactive(children=(IntSlider(value=0, description='t', max=20000), IntRangeSlider(value=(-20, 40), descript…